# BoltzGen minimal run from FASTA + epitope

This notebook generates a minimal BoltzGen YAML using only a target FASTA and epitope residue positions, then runs `boltzgen check` and `boltzgen run`.

Recommended environment creation:
```bash
conda env create -f environment.yml
conda activate boltzgen-fasta-epitope
```


In [ ]:
from pathlib import Path
import subprocess
import yaml

FASTA_PATH = 'target.fasta'
EPITOPE = '25-40'
OUTDIR = 'workbench_fasta_epitope'
PEPTIDE_LENGTH = '12..18'
PROTOCOL = 'peptide-anything'
NUM_DESIGNS = 100
BUDGET = 20
CACHE = None
CYCLIC = False


In [ ]:
def read_fasta(path):
    seq = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('>'):
                continue
            seq.append(line)
    sequence = ''.join(seq).replace(' ', '').upper()
    if not sequence:
        raise ValueError(f'No sequence found in FASTA: {path}')
    valid = set('ACDEFGHIKLMNPQRSTVWY')
    bad = sorted(set(sequence) - valid)
    if bad:
        raise ValueError(f"Invalid FASTA characters: {''.join(bad)}")
    return sequence

def parse_epitope(spec, seq_len):
    spec = spec.strip().replace('..', '-')
    positions = set()
    for part in spec.split(','):
        token = part.strip()
        if not token:
            continue
        if '-' in token:
            a, b = token.split('-', 1)
            a, b = int(a), int(b)
            if a > b:
                a, b = b, a
            for i in range(a, b + 1):
                positions.add(i)
        else:
            positions.add(int(token))
    if min(positions) < 1 or max(positions) > seq_len:
        raise ValueError(f'Epitope positions must be within 1..{seq_len}')
    return sorted(positions)

def build_binding_types(seq_len, positions):
    arr = ['u'] * seq_len
    for p in positions:
        arr[p - 1] = 'B'
    return ''.join(arr)

def build_design_spec(target_sequence, epitope_spec, peptide_length='12..18', cyclic=False):
    positions = parse_epitope(epitope_spec, len(target_sequence))
    binding_types = build_binding_types(len(target_sequence), positions)
    peptide_entity = {'protein': {'id': 'P', 'sequence': peptide_length}}
    if cyclic:
        peptide_entity['protein']['cyclic'] = True
    return {
        'entities': [
            {'protein': {'id': 'A', 'sequence': target_sequence, 'binding_types': binding_types}},
            peptide_entity,
        ]
    }

target_sequence = read_fasta(FASTA_PATH)
spec = build_design_spec(target_sequence, EPITOPE, PEPTIDE_LENGTH, CYCLIC)
outdir = Path(OUTDIR)
outdir.mkdir(parents=True, exist_ok=True)
yaml_path = outdir / 'design_from_fasta_epitope.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(spec, f, sort_keys=False, allow_unicode=True)
print(yaml_path)
print(f'Target length: {len(target_sequence)} aa')
print(f'Epitope: {EPITOPE}')


In [ ]:
check_cmd = ['boltzgen', 'check', str(yaml_path)]
if CACHE:
    check_cmd += ['--cache', CACHE]
print(' '.join(check_cmd))
subprocess.run(check_cmd, check=True)


In [ ]:
run_cmd = [
    'boltzgen', 'run', str(yaml_path),
    '--output', str(outdir),
    '--protocol', PROTOCOL,
    '--num_designs', str(NUM_DESIGNS),
    '--budget', str(BUDGET),
]
if CACHE:
    run_cmd += ['--cache', CACHE]
print(' '.join(run_cmd))
subprocess.run(run_cmd, check=True)


Typical outputs are written under `OUTDIR/`, including filtered and ranked final designs.
